<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session3_Argument_Principle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import mpmath as mp
import math
import time

mp.mp.dps = 50

# Rectangle boundaries
A = mp.mpf("0.1")
B = mp.mpf("0.9")
T1 = mp.mpf("10")
T2 = mp.mpf("60")

print("Session 03: Argument Principle")
print("Precision:", mp.mp.dps, "decimal digits")
print("Rectangle:")
print(f"Real: {A} to {B}")
print(f"Imaginary: {T1} to {T2}")

Session 03: Argument Principle
Precision: 50 decimal digits
Rectangle:
Real: 0.1 to 0.9
Imaginary: 10.0 to 60.0


In [2]:
#Construct the contour
def contour_points(n):
    """
    Generate points around the rectangular contour.
    Each side receives n subdivisions.
    The contour is counterclockwise.
    """

    edges = [
        # Bottom: left to right
        lambda u: A + (B - A) * u + 1j * T1,

        # Right: bottom to top
        lambda u: B + 1j * (T1 + (T2 - T1) * u),

        # Top: right to left
        lambda u: B - (B - A) * u + 1j * T2,

        # Left: top to bottom
        lambda u: A + 1j * (T2 - (T2 - T1) * u)
    ]

    points = []

    for edge in edges:
        for k in range(n):
            u = mp.mpf(k) / n
            points.append(edge(u))

    return points


print("Contour construction ready.")
print("Four sides, counterclockwise orientation.")

Contour construction ready.
Four sides, counterclockwise orientation.


In [3]:
#Calculate the winding number
def calculate_winding(n):
    points = contour_points(n)

    # Evaluate zeta along the contour
    values = [mp.zeta(s) for s in points]

    if any(v == 0 for v in values):
        raise ValueError(
            "A sampled contour point is a zero. "
            "Change the contour boundaries."
        )

    # Phase changes between consecutive points,
    # including the final-to-first closing segment
    phase_changes = []

    for i in range(len(values)):
        v1 = values[i]
        v2 = values[(i + 1) % len(values)]

        phase_changes.append(mp.arg(v2 / v1))

    total_phase = mp.fsum(phase_changes)

    winding = total_phase / (2 * mp.pi)

    maximum_step = max(abs(x) for x in phase_changes)

    minimum_sampled_magnitude = min(abs(v) for v in values)

    nearest_integer = int(mp.nint(winding))

    error = abs(winding - nearest_integer)

    return {
        "n": n,
        "winding": winding,
        "count": nearest_integer,
        "error": error,
        "max_phase_step": maximum_step,
        "min_boundary_abs": minimum_sampled_magnitude
    }


print("Winding-number calculation ready.")


Winding-number calculation ready.


In [4]:
#Refine the calculation
sampling_levels = [64, 128, 256, 512, 1024]

results = []

for n in sampling_levels:
    start = time.time()

    result = calculate_winding(n)

    elapsed = time.time() - start

    results.append(result)

    print("\n--------------------------------")
    print("Subdivisions per side:", n)
    print("Total contour points:", 4 * n)

    print(
        "Winding number:",
        mp.nstr(result["winding"], 20)
    )

    print("Nearest integer:", result["count"])

    print(
        "Distance from integer:",
        mp.nstr(result["error"], 8)
    )

    print(
        "Maximum phase increment:",
        mp.nstr(result["max_phase_step"], 8)
    )

    print(
        "Minimum sampled |zeta(s)|:",
        mp.nstr(result["min_boundary_abs"], 8)
    )

    print("Time:", round(elapsed, 2), "seconds")


--------------------------------
Subdivisions per side: 64
Total contour points: 256
Winding number: 13.0
Nearest integer: 13
Distance from integer: 0.0
Maximum phase increment: 2.517501
Minimum sampled |zeta(s)|: 0.30682717
Time: 0.62 seconds

--------------------------------
Subdivisions per side: 128
Total contour points: 512
Winding number: 13.0
Nearest integer: 13
Distance from integer: 0.0
Maximum phase increment: 1.444821
Minimum sampled |zeta(s)|: 0.29553154
Time: 1.07 seconds

--------------------------------
Subdivisions per side: 256
Total contour points: 1024
Winding number: 13.0
Nearest integer: 13
Distance from integer: 0.0
Maximum phase increment: 0.74905118
Minimum sampled |zeta(s)|: 0.27135389
Time: 2.13 seconds

--------------------------------
Subdivisions per side: 512
Total contour points: 2048
Winding number: 13.0
Nearest integer: 13
Distance from integer: 0.0
Maximum phase increment: 0.37964647
Minimum sampled |zeta(s)|: 0.27135389
Time: 7.03 seconds

----------

In [6]:
#Check numerical stability
if len(results) >= 2:
    last = results[-1]
    previous = results[-2]

    same_count = last["count"] == previous["count"]

    close_to_integer = (
        last["error"] < mp.mpf("1e-8")
        and previous["error"] < mp.mpf("1e-8")
    )

    small_phase_steps = (
        last["max_phase_step"] < mp.pi / 4
        and previous["max_phase_step"] < mp.pi / 4
    )

    print("NUMERICAL STABILITY CHECK")
    print("-------------------------")
    print("Same integer count:", same_count)
    print("Close to integer:", close_to_integer)
    print("Phase steps sufficiently small:", small_phase_steps)

    if same_count and close_to_integer and small_phase_steps:
        print("\nResult: STABLE NUMERICAL COUNT")
        print("Estimated number of zeros:", last["count"])
    else:
        print("\nResult: NOT YET STABLE")
        print("Increase sampling or investigate the contour.")

NUMERICAL STABILITY CHECK
-------------------------
Same integer count: True
Close to integer: True
Phase steps sufficiently small: True

Result: STABLE NUMERICAL COUNT
Estimated number of zeros: 13


In [ ]:
#Compare against known zeros
known_inside = []

for n in range(1, 101):
    rho = mp.zetazero(n)

    if (
        A < mp.re(rho) < B
        and T1 < mp.im(rho) < T2
    ):
        known_inside.append((n, rho))

print("KNOWN ZEROS INSIDE THE RECTANGLE")
print("--------------------------------")

for n, rho in known_inside:
    print(
        f"Zero {n}: "
        f"{mp.nstr(mp.re(rho), 12)} "
        f"+ {mp.nstr(mp.im(rho), 25)}i"
    )

print("\nKnown zeros in region:", len(known_inside))

if results:
    print("Numerical contour count:", results[-1]["count"])

    print(
        "Counts agree:",
        results[-1]["count"] == len(known_inside)
    )